# WP40 — Learned Safety Model: Neural Gödelian Governor (v0.4)
## SafetyClassifier · FailureMemory · LearnedSafetyGuard

Demonstrates **WP40**: replacing WP27's hard-coded Hoare invariants with a *learned* neural safety classifier that predicts P(safe | state, action) from historical modification outcomes.

> *"The centrencephalic system must be able to learn what constitutes a safe self-modification."* — Good (1965)

Runtime: **~3 min** (no GPU)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')); 
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np, matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp40_learned_safety import (
    ModificationRecord, SafetyDataset, SafetyClassifier,
    LearnedSafetyGuard, SafetyDecisionRecord, verify_wp40_exit_criteria
)
from prometheus.wp17_crls_synthesis import SynthesisAction
from prometheus.wp27_formal_invariants import make_default_registry, InvariantGuard
N_ACTIONS = len(SynthesisAction)
print(f'WP40 imports OK  n_actions={N_ACTIONS}')
print(f'SynthesisActions: {[a.name for a in SynthesisAction]}')

In [ ]:
# SafetyDataset — build labelled training set
import random
rng = random.Random(42)
D_STATE = 8
dataset = SafetyDataset(n_actions=N_ACTIONS, test_fraction=0.20)

# Generate 80 modification records
# Safe: accuracy stayed high after action
# Unsafe: accuracy dropped significantly
for i in range(80):
    acc_before = rng.uniform(0.55, 0.90)
    action_id  = rng.randint(0, N_ACTIONS-1)
    # DEMOTE_WORST and RESET_UNIFORM tend to be safer when acc is high
    if action_id in (0, 2) and acc_before > 0.65:
        acc_after = acc_before + rng.uniform(-0.03, 0.08)
        safe = True
    else:
        acc_after = acc_before + rng.uniform(-0.12, 0.05)
        safe = acc_after >= acc_before - 0.05
    sv_before = [acc_before, 0.2, 0.15, 1.5, 0.3, 0.1, i/80, 0.5]
    sv_after  = [acc_after,  0.2, 0.15, 1.4, 0.3, 0.1, (i+1)/80, 0.5]
    dataset.add(ModificationRecord(sv_before, action_id, sv_after, safe, generation=i))

bal = dataset.class_balance()
print(f'Dataset: {len(dataset)} records  safe={bal["safe"]}  unsafe={bal["unsafe"]}')

In [ ]:
# SafetyClassifier — train online
clf = SafetyClassifier(d_state=D_STATE, n_actions=N_ACTIONS, d_hidden=16, lr=0.05, seed=0)
train_recs, test_recs = dataset.train_test_split()

EPOCHS = 30
losses = []
for epoch in range(EPOCHS):
    batch = dataset.sample_batch(16)
    fl = [(r.to_features(N_ACTIONS), int(r.safe)) for r in batch]
    loss = clf.train_step(fl)
    losses.append(loss)
    if (epoch+1) % 5 == 0:
        ev = clf.evaluate(test_recs)
        print(f'  Epoch {epoch+1:3d}  loss={loss:.4f}  test_acc={ev["accuracy"]:.3f}  n={ev["n"]}')

print(f'\nFinal test accuracy: {clf.evaluate(test_recs)["accuracy"]:.3f}')

In [ ]:
# LearnedSafetyGuard — gate synthesis actions
fg    = InvariantGuard(make_default_registry())
guard = LearnedSafetyGuard(clf, fg, safety_threshold=0.60, abstain_threshold=0.20, min_train_samples=20)

print('Safety gate decisions:')
print(f'  {"State (acc)":>12} {"Action":<20} {"p_safe":>7} {"Decision":<20} Source')
print('-' * 75)

test_cases = [
    ([0.85, 0.2, 0.15, 1.5, 0.35, 0.12, 0.5, 0.5], SynthesisAction.DEMOTE_WORST),
    ([0.85, 0.2, 0.15, 1.5, 0.35, 0.12, 0.5, 0.5], SynthesisAction.PROMOTE_BEST),
    ([0.40, 0.2, 0.15, 2.0, 0.20, 0.05, 0.5, 0.5], SynthesisAction.RESET_UNIFORM),
    ([0.55, 0.2, 0.15, 1.8, 0.25, 0.08, 0.5, 0.5], SynthesisAction.BOOST_UPWARD),
]
safety_records = []
for sv, action in test_cases:
    aid = action.value if hasattr(action,'value') else 0
    oh  = [0.0]*N_ACTIONS; oh[aid%N_ACTIONS] = 1.0
    feat = sv + oh
    eff, src, p_safe = guard.apply(feat, len(dataset), {}, action)
    blocked = (eff != action)
    rec = SafetyDecisionRecord(len(safety_records), action, eff, src, p_safe, blocked, len(dataset), clf.train_steps)
    safety_records.append(rec)
    dec = 'BLOCK -> '+eff.name if blocked else 'ALLOW'
    print(f'  {sv[0]:>12.2f} {action.name:<20} {p_safe:>7.3f} {dec:<20} {src}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: training loss curve
ax = axes[0]
ax.plot(range(1, len(losses)+1), losses, 'b-', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('Safety Classifier Training Loss', fontweight='bold')

# Panel B: p_safe per test case
ax2 = axes[1]
labels = [a.name for _, a in test_cases]
p_safes = [r.p_safe for r in safety_records]
colors  = ['#4CAF50' if not r.blocked else '#F44336' for r in safety_records]
ax2.bar(range(len(labels)), p_safes, color=colors, edgecolor='black', alpha=0.85)
ax2.axhline(0.60, color='orange', linestyle='--', label='Safety threshold (0.60)')
ax2.set_xticks(range(len(labels))); ax2.set_xticklabels(labels, rotation=20, ha='right')
ax2.set_ylabel('P(safe)'); ax2.set_ylim(0, 1.1)
ax2.set_title('Safety Gate: P(safe) per Action', fontweight='bold'); ax2.legend()

# Panel C: decision source breakdown
ax3 = axes[2]
src_counts = {}
for r in safety_records:
    src_counts[r.decision_source] = src_counts.get(r.decision_source, 0) + 1
if src_counts:
    ax3.bar(src_counts.keys(), src_counts.values(), color='#9C27B0', edgecolor='black', alpha=0.85)
ax3.set_ylabel('Count'); ax3.set_title('Decision Source Breakdown', fontweight='bold')

fig.suptitle('WP40: Learned Safety Model — Neural Gödelian Governor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp40_learned_safety.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved wp40_learned_safety.png')

In [ ]:
criteria = verify_wp40_exit_criteria(clf, dataset, guard, safety_records)
print('WP40 Exit Criteria Verification'); print('='*62)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()): print('\nAll WP40 exit criteria satisfied.')

---
## Conclusions

**WP40** replaces brittle rule-based safety constraints with a learned neural governor:
- `SafetyClassifier` (2-layer MLP) trains online on historical (state, action, outcome) triples
- `LearnedSafetyGuard` defers to WP27's formal invariants when confidence is low — safe-by-default
- Blocks low-P(safe) actions; logs all decisions for audit

### Relation to WP27
| | WP27 InvariantGuard | WP40 LearnedSafetyGuard |
|---|---|---|
| **Rules** | Hand-written Hoare predicates | Learned from data |
| **Uncertainty** | None (hard block) | Abstain when uncertain |
| **Generalises** | To predicate violations only | To novel state combinations |
| **Updates** | Static (manual edit) | Online gradient descent |

### References
- Amodei et al. (2016) *Concrete Problems in AI Safety*
- Leike et al. (2018) *AI Safety Gridworlds*
- Good (1965) — centrencephalic safety governor must be able to learn